# PyTorch vs. PyTorch Lightning — A Hands-On Colab Tutorial

This notebook teaches the difference between **plain PyTorch** and **PyTorch Lightning** by solving the **same machine-learning problem twice**.

We will train a small neural network to learn:

\[
y = 2x + 1
\]

The goal is not the math itself. The goal is to see **who is responsible for the training loop and training infrastructure**.

By the end of this notebook, you should understand:

- What PyTorch provides
- What a manual PyTorch training loop looks like
- What PyTorch Lightning adds
- Which parts of the code Lightning removes or organizes
- Why Lightning becomes more useful as projects grow


## Cell 1 — Check the Colab environment

Google Colab usually comes with PyTorch preinstalled. This cell imports PyTorch and prints the version and available accelerator.

If Colab has a GPU enabled, `torch.cuda.is_available()` will return `True`.

You can enable a GPU in Colab using:

**Runtime → Change runtime type → Hardware accelerator → GPU**


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU


## Cell 2 — Install PyTorch Lightning

PyTorch Lightning is distributed through the `lightning` Python package.

We install it here so the second half of the notebook can use the Lightning `Trainer` and `LightningModule`.


In [2]:
!pip -q install lightning


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 37.4 MB/s eta 0:00:00


## Cell 3 — Import the libraries

We use the same PyTorch building blocks in both implementations:

- `torch` for tensors
- `torch.nn` for neural-network layers
- `TensorDataset` and `DataLoader` for batching

Later, we will also import `lightning` for the Lightning version.


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# Part 1 — Plain PyTorch

With plain PyTorch, you define both:

1. **The model**
2. **The training process**

That means you explicitly write the epoch loop, batch loop, gradient reset, backpropagation, and optimizer update.


## Cell 4 — Create a synthetic dataset

We generate 1,000 values of `x` between -10 and 10.

For every value of `x`, the correct target is:

\[
y = 2x + 1
\]

The neural network will learn this relationship from the examples.


In [4]:
X = torch.linspace(-10, 10, 1000).reshape(-1, 1)
y = 2 * X + 1

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst five examples:")
for i in range(5):
    print(f"x = {X[i].item():.3f}, y = {y[i].item():.3f}")


X shape: torch.Size([1000, 1])
y shape: torch.Size([1000, 1])

First five examples:
x = -10.000, y = -19.000
x = -9.980, y = -18.960
x = -9.960, y = -18.920
x = -9.940, y = -18.880
x = -9.920, y = -18.840


## Cell 5 — Put the data into a DataLoader

A `TensorDataset` pairs each input `X` with its target `y`.

A `DataLoader` then:

- splits the data into mini-batches
- shuffles the training data
- feeds batches to the model during training

Here we use a batch size of 32.


In [5]:
dataset = TensorDataset(X, y)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print("Number of batches:", len(train_loader))


Number of batches: 32


## Cell 6 — Define the neural network

This is a standard PyTorch `nn.Module`.

The model has:

- one input feature
- a hidden layer with 16 neurons
- a ReLU activation
- one output value

The `forward()` method describes how an input travels through the network.


In [6]:
class SimplePyTorchModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


pytorch_model = SimplePyTorchModel()

print(pytorch_model)


SimplePyTorchModel(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)


## Cell 7 — Define the loss function and optimizer

The **loss function** tells us how wrong the model's predictions are.

We use Mean Squared Error:

`nn.MSELoss()`

The **optimizer** updates the model's parameters to reduce the loss.

We use Adam with a learning rate of `0.01`.


In [7]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    pytorch_model.parameters(),
    lr=0.01
)

print("Loss function:", loss_fn)
print("Optimizer:", optimizer.__class__.__name__)


Loss function: MSELoss()
Optimizer: Adam


## Cell 8 — Train the model manually with PyTorch

This is the most important PyTorch cell in the notebook.

With plain PyTorch, **you own the training loop**.

For every batch, we explicitly perform:

1. Forward pass
2. Loss calculation
3. `optimizer.zero_grad()`
4. `loss.backward()`
5. `optimizer.step()`

These are the mechanics that PyTorch Lightning will later manage for us.


In [8]:
epochs = 20

for epoch in range(epochs):

    pytorch_model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        # 1. Forward pass
        predictions = pytorch_model(X_batch)

        # 2. Calculate loss
        loss = loss_fn(predictions, y_batch)

        # 3. Clear old gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update model parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:02d}/{epochs} "
        f"- Loss: {average_loss:.6f}"
    )


Epoch 01/20 - Loss: 58.776290
Epoch 02/20 - Loss: 1.164462
Epoch 03/20 - Loss: 0.311380
Epoch 04/20 - Loss: 0.194815
Epoch 05/20 - Loss: 0.135722
Epoch 06/20 - Loss: 0.093130
Epoch 07/20 - Loss: 0.067445
Epoch 08/20 - Loss: 0.048830
Epoch 09/20 - Loss: 0.036572
Epoch 10/20 - Loss: 0.027227
Epoch 11/20 - Loss: 0.020694
Epoch 12/20 - Loss: 0.015355
Epoch 13/20 - Loss: 0.011652
Epoch 14/20 - Loss: 0.009173
Epoch 15/20 - Loss: 0.007253
Epoch 16/20 - Loss: 0.005928
Epoch 17/20 - Loss: 0.004997
Epoch 18/20 - Loss: 0.004363
Epoch 19/20 - Loss: 0.003916
Epoch 20/20 - Loss: 0.003359


## Cell 9 — Test the PyTorch model

The true equation is:

\[
y = 2x + 1
\]

So for:

`x = 5`

we expect:

`y = 11`

We disable gradient tracking with `torch.no_grad()` because we are only making a prediction, not training.


In [9]:
pytorch_model.eval()

x_test = torch.tensor([[5.0]])

with torch.no_grad():
    prediction = pytorch_model(x_test)

print("PyTorch prediction:", prediction.item())
print("Expected value:", 11)


PyTorch prediction: 10.987927436828613
Expected value: 11


## What did PyTorch make us manage?

The key part of the previous implementation was the training loop:

```python
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

This gives you maximum control.

But as projects grow, the training code also has to deal with things such as:

- validation
- testing
- GPUs
- multiple GPUs
- distributed training
- mixed precision
- logging
- checkpoints
- early stopping
- restarting failed training jobs

That is where PyTorch Lightning becomes useful.


# Part 2 — Rewrite the Same Program with PyTorch Lightning

We will now solve the **same problem with the same PyTorch layers and optimizer**.

The main difference is that Lightning will own the outer training machinery.


## Cell 10 — Import Lightning

The modern Lightning package is imported as:

```python
import lightning as L
```

A Lightning model subclasses `L.LightningModule` instead of `nn.Module`.


In [10]:
import lightning as L

print("Lightning version:", L.__version__)


Lightning version: 2.6.5


## Cell 11 — Define the Lightning model

Notice that we are **still using PyTorch** inside the model:

- `nn.Linear`
- `nn.ReLU`
- `nn.MSELoss`
- `torch.optim.Adam`

Lightning does not replace PyTorch.

Instead, it gives the training code a standard structure.

The important methods are:

- `forward()` — defines inference
- `training_step()` — defines what happens for one training batch
- `configure_optimizers()` — tells Lightning which optimizer to use


In [11]:
class SimpleLightningModel(L.LightningModule):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.loss_fn = nn.MSELoss()


    def forward(self, x):
        return self.network(x)


    def training_step(self, batch, batch_idx):

        X_batch, y_batch = batch

        predictions = self(X_batch)

        loss = self.loss_fn(
            predictions,
            y_batch
        )

        self.log(
            "train_loss",
            loss,
            prog_bar=True
        )

        return loss


    def configure_optimizers(self):

        return torch.optim.Adam(
            self.parameters(),
            lr=0.01
        )


lightning_model = SimpleLightningModel()

print(lightning_model)


SimpleLightningModel(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
  (loss_fn): MSELoss()
)


## Cell 12 — Create a Lightning Trainer

This is where the major architectural difference appears.

In plain PyTorch, we wrote:

```python
for epoch in range(...):
    for batch in train_loader:
        ...
```

With Lightning, the `Trainer` manages that orchestration.

We only tell it how many epochs to train.


In [12]:
trainer = L.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=10
)


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Cell 13 — Train with Lightning

Training is now started with one line:

```python
trainer.fit(model, train_loader)
```

Internally, Lightning performs the batch loop and handles:

- gradient reset
- backpropagation
- optimizer updates
- device placement
- training state
- logging infrastructure

Your `training_step()` defines **what one batch means**.

The `Trainer` decides **how the overall training process runs**.


In [13]:
trainer.fit(
    lightning_model,
    train_dataloaders=train_loader
)


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network │ Sequential │     49 │ train │     0 │
│ 1 │ loss_fn │ MSELoss    │      0 │ train │     0 │
└───┴─────────┴────────────┴────────┴───────┴───────┘

Trainable params: 49                                                                                               
Non-trainable params: 0                                                                                            
Total params: 49                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


## Cell 14 — Test the Lightning model

Inference still looks like normal PyTorch.

That is another important point:

**A Lightning model is still a PyTorch model.**


In [14]:
lightning_model.eval()

x_test = torch.tensor([[5.0]])

# Move the input to the same device as the model
x_test = x_test.to(lightning_model.device)

with torch.no_grad():
    prediction = lightning_model(x_test)

print("Lightning prediction:", prediction.item())
print("Expected value:", 11)


Lightning prediction: 10.984942436218262
Expected value: 11


# Side-by-Side Comparison

The neural-network logic is nearly identical in both implementations.

The biggest difference is the **training infrastructure**.

| Responsibility | Plain PyTorch | PyTorch Lightning |
|---|---|---|
| Define layers | You | You |
| Define `forward()` | You | You |
| Define loss | You | You |
| Define optimizer | You | You |
| Define one training step | You | You |
| Epoch loop | You | Lightning |
| Batch loop | You | Lightning |
| `zero_grad()` | You | Lightning |
| `backward()` | You | Lightning |
| `optimizer.step()` | You | Lightning |
| GPU placement | Mostly you | Trainer |
| Multi-GPU | More code | Trainer configuration |
| Mixed precision | More code | Trainer configuration |
| Logging | You configure it | Integrated |
| Checkpoints | You build/configure | Integrated |


# The Core Difference in Code

### Plain PyTorch

You explicitly control the training mechanics:

```python
for epoch in range(epochs):

    for X_batch, y_batch in train_loader:

        predictions = model(X_batch)

        loss = loss_fn(
            predictions,
            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
```

### PyTorch Lightning

You describe a single training step:

```python
def training_step(self, batch, batch_idx):

    X, y = batch

    predictions = self(X)

    loss = self.loss_fn(
        predictions,
        y
    )

    return loss
```

Then Lightning controls the outer loop:

```python
trainer.fit(model, train_loader)
```


# Mental Model

A useful way to think about the relationship is:

```text
PyTorch
│
├── tensors
├── neural-network layers
├── automatic differentiation
├── optimizers
└── low-level training primitives
```

PyTorch Lightning sits above those primitives:

```text
PyTorch Lightning
│
├── training loop orchestration
├── validation/testing loops
├── device management
├── logging
├── checkpoints
├── mixed precision
├── multi-GPU
└── distributed training
        │
        ▼
      PyTorch
```

So:

> **PyTorch gives you the machine-learning building blocks. PyTorch Lightning gives you a structured training system around those PyTorch building blocks.**


# When Should You Use Each?

### Use plain PyTorch when:

- you are learning how neural-network training works
- you need unusual or highly customized training behavior
- you want complete control over every training operation
- your experiment is small enough that infrastructure code is manageable

### Use PyTorch Lightning when:

- your models are becoming production-scale
- you want cleaner and more standardized training code
- you need GPUs or multiple GPUs
- you need checkpointing and experiment logging
- you want to reduce training boilerplate
- multiple engineers need to work on the same training code


# Optional Exercise

Try changing the equation from:

\[
y = 2x + 1
\]

to:

\[
y = 5x - 3
\]

Change this line:

```python
y = 2 * X + 1
```

to:

```python
y = 5 * X - 3
```

Then retrain both models.

For `x = 5`, the expected answer becomes:

\[
5(5)-3 = 22
\]

If both models learn approximately `22`, you have confirmed that both frameworks are performing the same underlying machine-learning task.


# Next Step

A natural next tutorial is:

```text
PyTorch
   ↓
PyTorch Lightning
   ↓
Hugging Face Transformers
```

Using a real example such as a **customer-support ticket classifier**, we can compare:

1. a neural network written directly in PyTorch
2. the same training workflow organized with Lightning
3. a pretrained Hugging Face transformer such as DistilBERT

That makes the role of each technology much clearer in a real-world AI application.
